# Partie III — RNN, LSTM, GRU et Seq2Seq
## Modélisation de séquences et traduction sur données textuelles réelles

**Corpus retenu : *Tatoeba allemand → anglais (deu-eng)*** (sous-ensemble plafonné). Ce corpus parallèle
réel sert **à la fois** au modèle de langage (perplexité, comparaison RNN/LSTM/GRU) **et** au système
Seq2Seq de traduction (BLEU, décodage glouton et *beam search*). Il se distingue des exemples proposés
(IMDb, Tatoeba fra-eng).

> **Google Colab** : activer le GPU. Téléchargement automatique du corpus depuis le notebook.

### Plan
1. Objectif probabiliste d'un modèle de langage, règle de chaîne, perplexité
2. Préparation des données (tokenisation, vocabulaire, tokens spéciaux, padding, masquage, mini-lots)
3. RNN / LSTM / GRU : implémentation et comparaison (perplexité, stabilité, coût)
4. BPTT et effet du *gradient clipping*
5. Système Seq2Seq encodeur–décodeur + teacher forcing
6. Décodage glouton et *beam search*
7. Évaluation BLEU
8. Analyse critique, question de synthèse, question transversale finale


## 1. Modèle de langage : objectif probabiliste et perplexité

Un modèle de langage estime la probabilité d'une séquence de tokens $w_1, \dots, w_T$. Par la **règle de
la chaîne** :

$$ P(w_1, \dots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, \dots, w_{t-1}) $$

On entraîne le modèle à maximiser la log-vraisemblance, c.-à-d. à minimiser l'entropie croisée moyenne.
La **perplexité** est l'exponentielle de cette entropie croisée moyenne :

$$ \text{PPL} = \exp\!\left( -\frac{1}{T}\sum_{t=1}^{T} \log P(w_t \mid w_{<t}) \right) $$

Interprétation : une perplexité de $V$ signifie que le modèle hésite « comme s'il choisissait uniformément
parmi $V$ mots ». Plus elle est basse, mieux le modèle prédit la suite.


In [ ]:
%pip install -q sacrebleu

import os, re, time, math, random, zipfile, urllib.request
from collections import Counter
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def try_gpu(i=0):
    if torch.cuda.is_available() and torch.cuda.device_count() >= i + 1:
        return torch.device(f"cuda:{i}")
    return torch.device("cpu")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print("PyTorch :", torch.__version__, "| Colab :", IN_COLAB)


In [ ]:
# --- CONFIG centralisée (non hardcodé) ---
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    seed: int = 42
    device: torch.device = field(default_factory=try_gpu)
    data_dir: str = "/content/p3_data" if ("google.colab" in str(globals().get("get_ipython", lambda: "")())) else "p3_data"
    # Corpus
    max_pairs: int = 12000       # plafond du nombre de phrases
    max_len: int = 12            # longueur max (en tokens) des phrases conservées
    min_freq: int = 2            # fréquence minimale pour entrer dans le vocabulaire
    # Modèles
    embed_dim: int = 128
    hidden_dim: int = 256
    batch_size: int = 64
    # Entraînement modèle de langage
    lm_epochs: int = 6
    lm_lr: float = 3e-3
    # Entraînement Seq2Seq
    s2s_epochs: int = 12
    s2s_lr: float = 1e-3
    teacher_forcing: float = 0.5
    grad_clip: float = 1.0
    # Décodage
    beam_width: int = 3

CFG = Config()
os.makedirs(CFG.data_dir, exist_ok=True)
set_seed(CFG.seed)
print("Device :", CFG.device)


## 2. Préparation des données

Téléchargement automatique du corpus parallèle deu-eng (format `anglais \t allemand \t attribution`).
On tokenise au niveau **mot** (minuscules), on filtre par longueur, on construit deux **vocabulaires**
(source allemand, cible anglais) avec les tokens spéciaux `<pad>`, `<bos>`, `<eos>`, `<unk>`, puis on
prépare des mini-lots avec **padding** et **masquage**.


In [ ]:
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD, BOS, EOS, UNK = 0, 1, 2, 3

def tokenize(text):
    text = text.lower().strip()
    return re.findall(r"[a-zàâäéèêëîïôöùûüçß']+|[.,!?;]", text)

BROWSER_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "identity",
}

def _from_manythings():
    "Source principale : corpus Tatoeba deu-eng (manythings)."
    zip_path = os.path.join(CFG.data_dir, "deu-eng.zip")
    txt_path = os.path.join(CFG.data_dir, "deu.txt")
    if not os.path.exists(txt_path):
        url = "https://www.manythings.org/anki/deu-eng.zip"
        req = urllib.request.Request(url, headers=BROWSER_HEADERS)
        with urllib.request.urlopen(req, timeout=60) as r, open(zip_path, "wb") as f:
            f.write(r.read())
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(CFG.data_dir)
    pairs = []
    with open(txt_path, encoding="utf-8") as f:
        for line in f:
            parts = line.split("\t")
            if len(parts) >= 2:
                pairs.append((parts[1], parts[0]))   # (allemand, anglais)
    return pairs

def _from_hf(repo, config=None):
    "Repli via HuggingFace datasets (identifiants correctement namespacés)."
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "datasets"])
    from datasets import load_dataset
    ds = load_dataset(repo, config, split="train") if config else load_dataset(repo, split="train")
    pairs = []
    for ex in ds:
        if "translation" in ex:                       # ex. opus_books
            pairs.append((ex["translation"]["de"], ex["translation"]["en"]))
        else:                                          # ex. multi30k : colonnes 'de' / 'en'
            pairs.append((ex["de"], ex["en"]))
    return pairs

def download_corpus():
    "Essaie plusieurs sources en cascade ; retourne une liste de (allemand, anglais)."
    sources = [
        ("Tatoeba deu-eng (manythings)", _from_manythings),
        ("Multi30k de-en (HF)",          lambda: _from_hf("bentrevett/multi30k")),
        ("opus_books de-en (HF)",        lambda: _from_hf("Helsinki-NLP/opus_books", "de-en")),
    ]
    for name, fn in sources:
        try:
            pairs = fn()
            if pairs:
                print(f"Corpus chargé depuis : {name}  ({len(pairs)} paires)")
                return pairs
        except Exception as e:
            print(f"Échec '{name}' -> {str(e)[:90]}")
    raise RuntimeError("Aucune source de corpus disponible.")

raw_pairs = download_corpus()
print("Paires brutes :", len(raw_pairs))
print("Exemple (de, en) :", raw_pairs[0])


In [ ]:
# Tokenisation + filtrage par longueur + plafond
set_seed(CFG.seed)
random.shuffle(raw_pairs)

pairs = []
for de, en in raw_pairs:
    de_tok, en_tok = tokenize(de), tokenize(en)
    if 1 <= len(de_tok) <= CFG.max_len and 1 <= len(en_tok) <= CFG.max_len:
        pairs.append((de_tok, en_tok))
    if len(pairs) >= CFG.max_pairs:
        break
print("Paires retenues :", len(pairs))

class Vocab:
    "Vocabulaire token<->index avec fréquence minimale et tokens spéciaux."
    def __init__(self, token_lists, min_freq):
        counter = Counter(t for toks in token_lists for t in toks)
        self.itos = list(SPECIALS) + [t for t, c in counter.most_common() if c >= min_freq]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
    def __len__(self): return len(self.itos)
    def encode(self, tokens): return [self.stoi.get(t, UNK) for t in tokens]
    def decode(self, ids):
        return [self.itos[i] for i in ids if i not in (PAD, BOS, EOS)]

src_vocab = Vocab([p[0] for p in pairs], CFG.min_freq)   # allemand
tgt_vocab = Vocab([p[1] for p in pairs], CFG.min_freq)   # anglais
print(f"Vocab source (de) = {len(src_vocab)} | Vocab cible (en) = {len(tgt_vocab)}")


In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.data = []
        for de_tok, en_tok in pairs:
            src = src_vocab.encode(de_tok)
            tgt = [BOS] + tgt_vocab.encode(en_tok) + [EOS]
            self.data.append((torch.tensor(src), torch.tensor(tgt)))
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate(batch):
    "Padding des sources et cibles ; masque = positions non-pad."
    srcs, tgts = zip(*batch)
    src_pad = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD)
    tgt_pad = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD)
    return src_pad, tgt_pad

dataset = TranslationDataset(pairs, src_vocab, tgt_vocab)
n_val = max(1, int(0.1 * len(dataset)))
train_set, val_set = random_split(dataset, [len(dataset) - n_val, n_val],
                                  generator=torch.Generator().manual_seed(CFG.seed))
train_loader = DataLoader(train_set, batch_size=CFG.batch_size, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_set, batch_size=CFG.batch_size, shuffle=False, collate_fn=collate)
print(f"Train={len(train_set)}  Val={len(val_set)}")
src0, tgt0 = next(iter(train_loader))
print("Forme batch source/cible :", src0.shape, tgt0.shape)


## 3. RNN, LSTM, GRU — modèle de langage et comparaison

On entraîne un **modèle de langage** (prédiction du token suivant) sur la partie **anglaise** du corpus,
successivement avec une cellule **RNN**, **LSTM** puis **GRU**, et on compare leur **perplexité** de
validation, leur stabilité et leur coût.


In [ ]:
class RNNLanguageModel(nn.Module):
    "Modèle de langage récurrent ; cell in {rnn, lstm, gru}."
    def __init__(self, vocab_size, embed_dim, hidden_dim, cell="gru"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        rnns = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}
        self.rnn = rnns[cell](embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x):
        out, _ = self.rnn(self.embedding(x))
        return self.fc(out)

def lm_batches(loader):
    "Pour le LM : on utilise la séquence cible (anglais). entrée = seq[:-1], cible = seq[1:]."
    for _, tgt in loader:
        yield tgt[:, :-1], tgt[:, 1:]

def train_lm(cell, loader_train, loader_val, cfg, device, clip=None, track_grad=False):
    set_seed(cfg.seed)
    model = RNNLanguageModel(len(tgt_vocab), cfg.embed_dim, cfg.hidden_dim, cell).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lm_lr)
    crit = nn.CrossEntropyLoss(ignore_index=PAD)
    grad_norms, t0 = [], time.time()
    for ep in range(cfg.lm_epochs):
        model.train()
        for x, y in lm_batches(loader_train):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            loss.backward()
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),
                          clip if clip is not None else float("inf"))
            if track_grad:
                grad_norms.append(float(total_norm))
            opt.step()
    ppl = perplexity(model, loader_val, crit, device)
    return model, ppl, time.time() - t0, grad_norms

@torch.no_grad()
def perplexity(model, loader, crit, device):
    model.eval(); tot, n = 0.0, 0
    for x, y in lm_batches(loader):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        ntok = (y != PAD).sum().item()
        tot += loss.item() * ntok; n += ntok
    return math.exp(tot / max(n, 1))


In [ ]:
import pandas as pd
results = []
for cell in ["rnn", "lstm", "gru"]:
    model, ppl, dt, _ = train_lm(cell, train_loader, val_loader, CFG, CFG.device, clip=CFG.grad_clip)
    nparam = sum(p.numel() for p in model.parameters())
    results.append({"cellule": cell.upper(), "perplexité_val": round(ppl, 2),
                    "params": nparam, "temps_s": round(dt, 1)})
    print(f"{cell.upper():5s} -> PPL={ppl:.2f} | params={nparam} | {dt:.1f}s")
pd.DataFrame(results).set_index("cellule")


**Interprétation.** Le RNN simple souffre du gradient qui s'évanouit/explose sur les dépendances
longues : sa perplexité est généralement la plus élevée et son entraînement le moins stable. Le **LSTM**
(portes d'entrée/oubli/sortie) et le **GRU** (portes de mise à jour/réinitialisation) mémorisent mieux le
contexte ; le GRU atteint souvent une perplexité comparable au LSTM pour **moins de paramètres** et un
coût moindre — d'où son usage pour le Seq2Seq ci-dessous.


## 4. BPTT et effet du gradient clipping

La **rétropropagation à travers le temps (BPTT)** déroule le réseau sur la séquence : les gradients se
multiplient le long du temps, ce qui peut les faire **exploser**. Le *gradient clipping* borne la norme du
gradient et stabilise l'entraînement. On illustre l'effet en suivant la norme des gradients **avec** et
**sans** clipping.


In [ ]:
_, _, _, gn_noclip = train_lm("rnn", train_loader, val_loader, CFG, CFG.device, clip=None, track_grad=True)
_, _, _, gn_clip   = train_lm("rnn", train_loader, val_loader, CFG, CFG.device, clip=CFG.grad_clip, track_grad=True)

plt.figure(figsize=(9, 4))
plt.plot(gn_noclip, alpha=0.7, label="sans clipping")
plt.plot(gn_clip, alpha=0.7, label=f"avec clipping (max={CFG.grad_clip})")
plt.xlabel("itération"); plt.ylabel("norme du gradient"); plt.yscale("log")
plt.title("Effet du gradient clipping (BPTT, cellule RNN)"); plt.legend(); plt.show()
print("Norme max sans clipping :", round(max(gn_noclip), 2))
print("Norme max avec clipping :", round(max(gn_clip), 2))


## 5. Système Seq2Seq encodeur–décodeur

Architecture **encodeur–décodeur** à base de GRU. L'encodeur résume la phrase allemande en un état caché ;
le décodeur génère l'anglais token par token. À l'entraînement, on utilise le **teacher forcing** (on
fournit parfois le vrai token précédent plutôt que la prédiction).


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
    def forward(self, src):
        _, h = self.gru(self.embedding(src))
        return h                                 # [1, B, hidden]

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def step(self, inp, hidden):
        "Un pas de décodage. inp:[B,1] -> logits:[B,vocab], hidden."
        out, hidden = self.gru(self.embedding(inp), hidden)
        return self.fc(out.squeeze(1)), hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder
    def forward(self, src, tgt, teacher_forcing=0.5):
        hidden = self.encoder(src)
        B, T = tgt.shape
        inp = tgt[:, 0:1]                        # <bos>
        logits_seq = []
        for t in range(1, T):
            logits, hidden = self.decoder.step(inp, hidden)
            logits_seq.append(logits)
            if random.random() < teacher_forcing:
                inp = tgt[:, t:t + 1]
            else:
                inp = logits.argmax(1, keepdim=True)
        return torch.stack(logits_seq, dim=1)    # [B, T-1, vocab]

set_seed(CFG.seed)
seq2seq = Seq2Seq(Encoder(len(src_vocab), CFG.embed_dim, CFG.hidden_dim),
                  Decoder(len(tgt_vocab), CFG.embed_dim, CFG.hidden_dim)).to(CFG.device)
print(seq2seq)


In [ ]:
def train_seq2seq(model, loader, cfg, device):
    opt = torch.optim.Adam(model.parameters(), lr=cfg.s2s_lr)
    crit = nn.CrossEntropyLoss(ignore_index=PAD)
    model.train()
    for ep in range(cfg.s2s_epochs):
        run, n = 0.0, 0
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            opt.zero_grad()
            logits = model(src, tgt, teacher_forcing=cfg.teacher_forcing)
            loss = crit(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            run += loss.item() * src.size(0); n += src.size(0)
        if (ep + 1) % 2 == 0 or ep == 0:
            print(f"  epoch {ep+1}/{cfg.s2s_epochs} | loss {run/n:.4f}")

train_seq2seq(seq2seq, train_loader, CFG, CFG.device)


## 6. Décodage : glouton et *beam search*

- **Décodage glouton** : à chaque pas, on choisit le token le plus probable.
- **Beam search** : on conserve les `beam_width` séquences partielles les plus probables, ce qui explore
  mieux l'espace et améliore souvent la qualité.


In [ ]:
@torch.no_grad()
def greedy_decode(model, src, max_len, device):
    model.eval()
    hidden = model.encoder(src.to(device))
    inp = torch.tensor([[BOS]], device=device)
    out_ids = []
    for _ in range(max_len):
        logits, hidden = model.decoder.step(inp, hidden)
        nxt = logits.argmax(1).item()
        if nxt == EOS: break
        out_ids.append(nxt); inp = torch.tensor([[nxt]], device=device)
    return out_ids

@torch.no_grad()
def beam_search_decode(model, src, beam_width, max_len, device):
    model.eval()
    hidden = model.encoder(src.to(device))
    beams = [(0.0, [BOS], hidden)]               # (log-prob cumulée, séquence, état)
    completed = []
    for _ in range(max_len):
        candidates = []
        for score, seq, h in beams:
            if seq[-1] == EOS:
                completed.append((score, seq)); continue
            inp = torch.tensor([[seq[-1]]], device=device)
            logits, h2 = model.decoder.step(inp, h)
            logp = F.log_softmax(logits, dim=-1).squeeze(0)
            topv, topi = logp.topk(beam_width)
            for v, i in zip(topv.tolist(), topi.tolist()):
                candidates.append((score + v, seq + [i], h2))
        if not candidates: break
        beams = sorted(candidates, key=lambda x: x[0] / len(x[1]), reverse=True)[:beam_width]
    completed += [(s, seq) for s, seq, _ in beams]
    best = max(completed, key=lambda x: x[0] / len(x[1]))
    return [i for i in best[1] if i not in (BOS, EOS)]

def ids_to_text(ids, vocab):
    return " ".join(vocab.itos[i] for i in ids if i not in (PAD, BOS, EOS))

# Exemple qualitatif
src_ex, tgt_ex = val_set[0]
print("Source (de)   :", ids_to_text(src_ex.tolist(), src_vocab))
print("Référence (en):", ids_to_text(tgt_ex.tolist(), tgt_vocab))
print("Glouton       :", ids_to_text(greedy_decode(seq2seq, src_ex.unsqueeze(0), CFG.max_len, CFG.device), tgt_vocab))
print("Beam search   :", ids_to_text(beam_search_decode(seq2seq, src_ex.unsqueeze(0), CFG.beam_width, CFG.max_len, CFG.device), tgt_vocab))


## 7. Évaluation BLEU

On compare la qualité de traduction (décodage glouton vs *beam search*) avec le score **BLEU** (sacrebleu)
sur l'ensemble de validation.


In [ ]:
import sacrebleu

def evaluate_bleu(model, dataset, decode_fn, device, n=300):
    refs, hyps = [], []
    for i in range(min(n, len(dataset))):
        src, tgt = dataset[i]
        ref = ids_to_text(tgt.tolist(), tgt_vocab)
        hyp = ids_to_text(decode_fn(src.unsqueeze(0)), tgt_vocab)
        refs.append(ref); hyps.append(hyp)
    return sacrebleu.corpus_bleu(hyps, [refs]).score

bleu_greedy = evaluate_bleu(seq2seq, val_set,
    lambda s: greedy_decode(seq2seq, s, CFG.max_len, CFG.device), CFG.device)
bleu_beam = evaluate_bleu(seq2seq, val_set,
    lambda s: beam_search_decode(seq2seq, s, CFG.beam_width, CFG.max_len, CFG.device), CFG.device)
print(f"BLEU glouton    : {bleu_greedy:.2f}")
print(f"BLEU beam (w={CFG.beam_width}) : {bleu_beam:.2f}")


## 8. Analyse critique

- **RNN → LSTM/GRU.** La comparaison de perplexité montre l'apport du *gating* pour mémoriser le contexte
  et stabiliser le gradient (confirmé par l'expérience de clipping). Le GRU offre le meilleur compromis
  performance/coût, justifiant son emploi dans le Seq2Seq.
- **Seq2Seq.** Avec un corpus plafonné et un encodeur à état unique (sans attention), la traduction reste
  approximative (BLEU modeste) : le goulot d'étranglement de l'état caché limite les phrases longues.
  Le *beam search* améliore généralement le BLEU par rapport au glouton en explorant plusieurs hypothèses.
- **Limites.** Vocabulaire tronqué (`<unk>`), absence de mécanisme d'**attention**, corpus réduit pour
  rester exécutable. Ces limites pointent directement vers les architectures à attention / Transformers.

## Question de synthèse — Partie III

> *Dans quelle mesure les architectures récurrentes modélisent-elles efficacement une séquence réelle, et
> comment justifier le passage RNN → LSTM/GRU → encodeur–décodeur ?*

Les RNN factorisent $P(w_{1:T})$ token par token et capturent la dépendance temporelle, mais le RNN simple
peine sur les **dépendances longues** (gradient instable). Les cellules à portes (LSTM/GRU) introduisent une
**mémoire** régulée qui atténue ce problème — ce que confirment la perplexité plus basse et la stabilité du
gradient. Pour une tâche de **transduction** (traduction), une simple prédiction du token suivant ne suffit
plus : le schéma **encodeur–décodeur** sépare la compréhension de la phrase source de la génération de la
cible, et le décodage (glouton vs *beam search*) arbitre entre coût et qualité. Les limites observées
(état unique, BLEU modeste) motivent l'ajout de l'**attention**.

---

## Question transversale finale

> *Comment le deep learning adapte-t-il ses architectures à la structure des données — tabulaire, image et
> séquentielle — et pourquoi un même paradigme d'apprentissage supervisé doit-il être décliné différemment ?*

Les trois parties illustrent un même principe : **l'architecture encode les a priori adaptés à la géométrie
des données**.

| Donnée | Structure exploitée | Architecture | Mécanisme clé |
|---|---|---|---|
| Tabulaire (Partie I) | aucune structure spatiale/temporelle | **MLP** | couches denses, interactions non linéaires |
| Image (Partie II) | dépendance **locale** + invariance par translation | **CNN** | localité, partage de poids, hiérarchie |
| Séquence (Partie III) | **temporalité** + dépendances à distance | **RNN/LSTM/GRU/Seq2Seq** | récurrence, mémoire à portes, encodeur–décodeur |

Le paradigme reste l'**apprentissage supervisé par descente de gradient**, mais le *biais inductif* change :
le MLP ne suppose aucune structure, le CNN suppose la localité spatiale, les modèles récurrents supposent une
dépendance temporelle ordonnée. Choisir l'architecture, c'est **injecter la bonne hypothèse sur la donnée** —
ce qui réduit le nombre de paramètres nécessaires, accélère l'apprentissage et améliore la généralisation.
Quand ces hypothèses deviennent insuffisantes (dépendances très longues), on enrichit le modèle (attention,
Transformers), prolongeant la même logique d'adaptation de l'architecture à la structure des données.
